In [1]:
!pip install -q torchdiffeq pysindy scikit-learn scipy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchdiffeq import odeint

torch.manual_seed(42)
np.random.seed(42)

class PK_Model_GroundTruth(nn.Module):
    def forward(self, t, y):
        GI, Blood, Tissue = y[..., 0], y[..., 1], y[..., 2]

        k_a = 5.0
        k_12 = 0.5
        k_21 = 0.2
        V_max = 1.0
        K_m = 0.5

        liver_clearance = (V_max * Blood) / (K_m + Blood)

        d_GI = -k_a * GI
        d_Blood = k_a * GI - k_12 * Blood + k_21 * Tissue - liver_clearance
        d_Tissue = k_12 * Blood - k_21 * Tissue

        return torch.stack((d_GI, d_Blood, d_Tissue), dim=-1)

t_eval = torch.linspace(0, 10.0, 100)
y0 = torch.tensor([[2.0, 0.0, 0.0],[5.0, 0.0, 0.0], [1.0, 0.0, 0.0]])

with torch.no_grad():
    y_true = odeint(PK_Model_GroundTruth(), y0, t_eval, method='dopri5')
    noise_level = 0.05
    y_noisy = y_true + torch.randn_like(y_true) * noise_level * torch.max(y_true, dim=0).values
    y_noisy = torch.clamp(y_noisy, min=0.0)

class ChemODE_Bio(nn.Module):
    def __init__(self, n_species=3):
        super().__init__()
        self.n_terms = 3
        self.W_num = nn.Parameter(torch.randn(n_species, self.n_terms) * 0.1)
        self.W_den = nn.Parameter(torch.randn(n_species, self.n_terms) * 0.1)
        self.register_buffer('sigma_lib', torch.ones(self.n_terms))

    def build_library(self, y):
        return y

    def forward(self, t, y):
        y_safe = torch.clamp(y, min=0.0)
        Theta = self.build_library(y_safe)
        Theta_norm = Theta / (self.sigma_lib + 1e-6)

        numerator = Theta_norm @ self.W_num.t()
        den_blood = F.softplus(self.W_den[1, 1]) * Theta_norm[..., 1] + 1.0

        dydt = numerator.clone()
        dydt[..., 1] = numerator[..., 1] / den_blood

        return torch.clamp(dydt, -50.0, 50.0)

model = ChemODE_Bio()
with torch.no_grad():
    raw_Theta = model.build_library(y_noisy.reshape(-1, 3))
    model.sigma_lib = torch.clamp(torch.std(raw_Theta, dim=0).float(), min=1e-3)

opt = torch.optim.Adam(model.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=150, gamma=0.5)

epochs = 600
for epoch in range(epochs):
    opt.zero_grad()
    pred = odeint(model, y0, t_eval, method='rk4', options={'step_size': 0.05})

    loss = torch.mean((pred - y_noisy)**2)
    loss += 0.001 * torch.sum(torch.abs(model.W_num))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    scheduler.step()

with torch.no_grad():
    pred_final = odeint(model, y0, t_eval, method='rk4', options={'step_size': 0.01})

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

batch_idx = 1
y_noisy_batch = y_noisy[:, batch_idx, :].detach().numpy()
pred_batch = pred_final[:, batch_idx, :].detach().numpy()
t_np = t_eval.numpy()

fig, ax = plt.subplots(figsize=(8, 5))

species =["GI Tract", "Blood Plasma", "Peripheral Tissue"]
colors =["#e41a1c", "#377eb8", "#4daf4a"]
markers = ["o", "s", "^"]

for i in range(3):
    ax.scatter(t_np[::4], y_noisy_batch[::4, i],
               color=colors[i], marker=markers[i], s=25,
               alpha=0.6, label=f'Measured {species[i]}', zorder=3)
    ax.plot(t_np, pred_batch[:, i],
            color=colors[i], linewidth=2.0, linestyle='-',
            label=f'ChemODE {species[i]}', zorder=2)

ax.set_title("Neural Discovery of 3-Compartment PK Dynamics", fontweight='bold', pad=15)
ax.set_xlabel("Time (Hours Post-Dose)")
ax.set_ylabel("Drug Concentration (mg/L)")

ax.grid(True, linestyle='--', alpha=0.5, zorder=1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=True, facecolor='white', framealpha=0.9, loc='upper right')

plt.tight_layout()
plt.savefig("figure1.pdf", format='pdf', bbox_inches='tight')
plt.savefig("figure1.png", dpi=300, bbox_inches='tight')
plt.close()

In [2]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchdiffeq import odeint

torch.manual_seed(42)

t_exp = torch.tensor([0.0, 0.25, 0.57, 1.12, 2.02, 3.82, 5.1, 7.03, 9.05, 12.12, 24.37])
blood_exp = torch.tensor([0.0, 2.84, 6.57, 10.5, 8.33, 5.8, 5.09, 3.95, 2.53, 1.47, 0.0])
y0_exp = torch.tensor([[15.0, 0.0]])

class ChemODE_RealPK(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = nn.Parameter(torch.randn(2, 2) * 0.1)

    def forward(self, t, y):
        y_safe = torch.clamp(y, min=0.0)
        return y_safe @ self.W.t()

model = ChemODE_RealPK()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=200, gamma=0.5)

for epoch in range(800):
    opt.zero_grad()
    pred = odeint(model, y0_exp, t_exp, method='dopri5').squeeze(1)

    loss = torch.mean((pred[:, 1] - blood_exp)**2)
    loss += 0.1 * torch.sum(torch.abs(model.W))

    loss.backward()
    opt.step()
    scheduler.step()

with torch.no_grad():
    t_dense = torch.linspace(0, 25.0, 100)
    pred_dense = odeint(model, y0_exp, t_dense, method='dopri5').squeeze(1)

plt.rcParams.update({
    "font.family": "serif",
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(7, 5))
plt.scatter(t_exp.numpy(), blood_exp.numpy(), color='red', label='Real Human Data (Assay)', zorder=5)
plt.plot(t_dense.numpy(), pred_dense[:, 1].numpy(), color='blue', linewidth=2.5, label='ChemODE Fit')

plt.title("Theophylline Pharmacokinetics: ChemODE Validation", fontsize=14, fontweight='bold')
plt.xlabel("Time (Hours)", fontsize=12)
plt.ylabel("Blood Concentration (mg/L)", fontsize=12)
plt.legend(frameon=True, fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("figure2.pdf", format='pdf', bbox_inches='tight')
plt.savefig("figure2.png", dpi=300, bbox_inches='tight')
plt.close()

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchdiffeq import odeint

def run_enzyme_trial(seed, use_softplus=True):
    torch.manual_seed(seed)

    t_eval = torch.linspace(0, 5.0, 50)
    y0 = torch.tensor([[5.0]])

    class TrueEnzyme(nn.Module):
        def forward(self, t, y):
            return -(2.0 * y) / (0.5 + y)

    with torch.no_grad():
        y_true = odeint(TrueEnzyme(), y0, t_eval, method='dopri5')
        y_noisy = y_true + torch.randn_like(y_true) * 0.02 * y_true.max()

    class EnzymeSurrogate(nn.Module):
        def __init__(self, use_softplus):
            super().__init__()
            self.vmax_w = nn.Parameter(torch.tensor([-1.0]))
            self.km_w = nn.Parameter(torch.tensor([0.1]))
            self.use_softplus = use_softplus

        def forward(self, t, y):
            num = self.vmax_w * y
            den = (F.softplus(self.km_w) if self.use_softplus else self.km_w) + y
            return num / den

    model = EnzymeSurrogate(use_softplus)
    opt = torch.optim.Adam(model.parameters(), lr=0.1)

    for epoch in range(300):
        opt.zero_grad()
        pred = odeint(model, y0, t_eval, method='rk4', options={'step_size': 0.1})
        loss = torch.mean((pred - y_noisy)**2)
        loss.backward()
        opt.step()

    vmax_learned = abs(model.vmax_w.item())
    km_learned = F.softplus(model.km_w).item() if use_softplus else model.km_w.item()

    return vmax_learned, km_learned

seeds =[42, 12, 99, 1024, 2026, 7, 777, 888, 123, 456]

vmax_chemode, km_chemode = [],[]
for s in seeds:
    v, k = run_enzyme_trial(s, use_softplus=True)
    vmax_chemode.append(v)
    km_chemode.append(k)

vmax_abl, km_abl = [],[]
for s in seeds:
    v, k = run_enzyme_trial(s, use_softplus=False)
    vmax_abl.append(v)
    km_abl.append(k)

print("Table 1: 10-Seed Uncertainty Quantification and Ablation Study for Enzyme Kinetics.")
print("-" * 75)
print(f"{'Configuration':<35} | {'Parameter':<10} | {'True Target':<12} | {'Learned Mean ± SD'}")
print("-" * 75)
print(f"{'ChemODE (Softplus Constraint)':<35} | {'V_max':<10} | {'2.00':<12} | {np.mean(vmax_chemode):.3f} ± {np.std(vmax_chemode):.3f}")
print(f"{'':<35} | {'K_m':<10} | {'0.50':<12} | {np.mean(km_chemode):.3f} ± {np.std(km_chemode):.3f}")
print("-" * 75)
print(f"{'Ablation (Unconstrained Rational)':<35} | {'V_max':<10} | {'2.00':<12} | {np.mean(vmax_abl):.3f} ± {np.std(vmax_abl):.3f}")
print(f"{'':<35} | {'K_m':<10} | {'0.50':<12} | {np.mean(km_abl):.3f} ± {np.std(km_abl):.3f}")
print("-" * 75)

Table 1: 10-Seed Uncertainty Quantification and Ablation Study for Enzyme Kinetics.
---------------------------------------------------------------------------
Configuration                       | Parameter  | True Target  | Learned Mean ± SD
---------------------------------------------------------------------------
ChemODE (Softplus Constraint)       | V_max      | 2.00         | 1.974 ± 0.052
                                    | K_m        | 0.50         | 0.464 ± 0.086
---------------------------------------------------------------------------
Ablation (Unconstrained Rational)   | V_max      | 2.00         | 1.285 ± 0.423
                                    | K_m        | 0.50         | 0.292 ± 0.111
---------------------------------------------------------------------------


In [4]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchdiffeq import odeint
from scipy.interpolate import UnivariateSpline
from sklearn.linear_model import Lasso
import warnings

warnings.filterwarnings("ignore")

class Brusselator(nn.Module):
    def forward(self, t, y):
        if y.ndim == 1: y = y.unsqueeze(0)
        x, y_c = y[:, 0], y[:, 1]
        return torch.stack([1.0 + (x**2 * y_c) - 4.0*x, 3.0*x - (x**2 * y_c)], dim=1)

t_eval = torch.linspace(0, 40, 400)
y0 = torch.tensor([[1.0, 1.0]])

with torch.no_grad():
    y_true = odeint(Brusselator(), y0, t_eval, method='dopri5').squeeze(1)
    torch.manual_seed(42)
    y_noisy = y_true + torch.randn_like(y_true) * 0.05

t_np = t_eval.numpy()
y_np = y_noisy.numpy()

spline_x = UnivariateSpline(t_np, y_np[:, 0], s=1.0)
spline_y = UnivariateSpline(t_np, y_np[:, 1], s=1.0)

X_clean = np.stack([spline_x(t_np), spline_y(t_np)], axis=1)
dX_clean = np.stack([spline_x.derivative()(t_np), spline_y.derivative()(t_np)], axis=1)

def build_library_np(y):
    x = y[:, 0]
    yc = y[:, 1]
    lib = np.stack([x, yc, x**2, yc**2, x*yc, (x**2)*yc, np.ones_like(x)], axis=1)
    return lib

Theta = build_library_np(X_clean)

reg = Lasso(alpha=0.01, fit_intercept=False, max_iter=10000)
reg.fit(Theta, dX_clean)
W_initial = torch.tensor(reg.coef_, dtype=torch.float32)

class ChemODE_Final(nn.Module):
    def __init__(self, w_init):
        super().__init__()
        self.W = nn.Parameter(w_init)

    def compute_library(self, y):
        if y.ndim == 1: y = y.unsqueeze(0)
        x, y_c = y[:, 0:1], y[:, 1:2]
        return torch.cat([x, y_c, x**2, y_c**2, x*y_c, (x**2)*y_c, torch.ones_like(x)], dim=1)

    def forward(self, t, y):
        y = torch.relu(y)
        lib = self.compute_library(y)
        dydt = lib @ self.W.t()
        dydt = torch.clamp(dydt, -20.0, 20.0)
        return dydt

model = ChemODE_Final(W_initial)

with torch.no_grad():
    pred_full = odeint(model, y0, t_eval, method='rk4', options={'step_size':0.05}).squeeze(1)

plt.rcParams.update({
    "font.size": 12,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

plt.figure(figsize=(10, 5))
plt.plot(t_eval, y_true[:, 0], 'k--', linewidth=1.5, alpha=0.5, label='True X')
plt.plot(t_eval, y_true[:, 1], 'k:', linewidth=1.5, alpha=0.5, label='True Y')

plt.plot(t_eval, pred_full[:, 0], 'r-', linewidth=2.0, label='ChemODE X')
plt.plot(t_eval, pred_full[:, 1], 'c-', linewidth=2.0, label='ChemODE Y')

plt.title("Figure 3: Brusselator Reconstruction", fontsize=16)
plt.legend(loc='upper right', framealpha=0.95)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig('figure3.pdf', format='pdf', bbox_inches='tight')
plt.savefig('figure3.png', dpi=300, bbox_inches='tight')
plt.close()

In [16]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pysindy as ps
from torchdiffeq import odeint

# 1. Ground Truth System
class StiffSystem(nn.Module):
    def forward(self, t, y):
        if y.ndim == 1: y = y.unsqueeze(0)
        A, B = y[:, 0], y[:, 1]
        return torch.stack([-0.1*A, 0.1*A - 10.0*B], dim=1)

t_eval = torch.linspace(0, 2.0, 100)
y0 = torch.tensor([[1.0, 0.0]])

with torch.no_grad():
    y_true_tensor = odeint(StiffSystem(), y0, t_eval, method='rk4', options={'step_size':0.005}).squeeze(1)
    y_true_np = y_true_tensor.numpy()

# 2. ChemODE Model
class ChemODE_Stiff(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = nn.Parameter(torch.zeros(2, 2))
        self.register_buffer('sigma', torch.ones(2))

    def forward(self, t, y):
        if y.ndim == 1: y = y.unsqueeze(0)
        y_norm = y / self.sigma
        dydt = y_norm @ self.W.t()
        return dydt

# 3. Training Loop
def train_chemode(y_noisy_tensor, t_tensor):
    model = ChemODE_Stiff()

    with torch.no_grad():
        raw_std = torch.std(y_noisy_tensor, dim=0).float()
        model.sigma = torch.max(raw_std, torch.tensor(0.5))

    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    time_weights = torch.exp(-5.0 * t_tensor)

    for i in range(1000):
        opt.zero_grad()
        pred = odeint(model, y_noisy_tensor[0], t_tensor, method='rk4', options={'step_size':0.005})
        if pred.ndim == 3: pred = pred.squeeze(1)

        err_sq = (pred - y_noisy_tensor)**2
        loss = torch.mean(time_weights.unsqueeze(1) * err_sq)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

    W_phys = model.W.detach() / model.sigma
    return abs(W_phys[1, 1].item())

# 4. Comparison Benchmark
noise_levels = [0.01, 0.05, 0.10]
sindy_errors = []
chemode_errors = []


for noise in noise_levels:
    np.random.seed(42)
    noise_vals = np.random.normal(0, noise * np.abs(y_true_np).max(), y_true_np.shape)
    y_n_sindy = y_true_np + noise_vals

    smoother = ps.SmoothedFiniteDifference()
    model_sindy = ps.SINDy(differentiation_method=smoother)
    try:
        model_sindy.fit(y_n_sindy, t=t_eval.numpy(), feature_names=["A", "B"])
        coeffs = model_sindy.coefficients()
        k_pred_sindy = abs(coeffs[1, 2])
    except:
        k_pred_sindy = 0.0
    sindy_errors.append(abs(k_pred_sindy - 10.0))

    trial_errors = []
    for seed in range(10):
        torch.manual_seed(seed)
        np.random.seed(seed)
        y_n_torch = y_true_tensor + torch.randn_like(y_true_tensor) * noise * torch.abs(y_true_tensor).max()
        k_est = train_chemode(y_n_torch.float(), t_eval.float())
        trial_errors.append(abs(k_est - 10.0))

    median_err = np.median(trial_errors)
    chemode_errors.append(median_err)

    print(f"Noise {int(noise*100)}% | SINDy Err: {sindy_errors[-1]:.2f} | ChemODE Err: {chemode_errors[-1]:.2f}")

Noise 1% | SINDy Err: 10.00 | ChemODE Err: 5.91
Noise 5% | SINDy Err: 45.78 | ChemODE Err: 0.99
Noise 10% | SINDy Err: 68.15 | ChemODE Err: 0.72
